# 08 - Physics & Spatial Influence Pitch Control Model

## Executive Summary & Objectives
This notebook implements the spatial influence and pitch control model:
1. **Pitch Control Physics**: Evaluates the probability that a team controls any given location $(x, y)$ on the pitch.
2. **Time-to-Intercept Formulation**: Computes time required for players of both teams to reach location $(x, y)$, incorporating reaction time $t_{\text{reaction}} = 0.7\text{s}$ and maximum running speed $v_{\text{max}} = 5.0\text{ m/s}$:
   $$T_i(x, y) = t_{\text{reaction}} + \frac{\|(x, y) - (x_i, y_i)\|}{v_{\text{max}}}$$
3. **Spatial Sigmoid Probability**: Integrates relative arrival times using logistic link:
   $$P_{\text{Att}}(x, y) = \sigma\left(\frac{T_{\text{Def}}^{\min}(x, y) - T_{\text{Att}}^{\min}(x, y)}{\tau}\right)$$
4. **Visual Pitch Control Dominance Surface**.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)

DATA_DIR = '../data' if os.path.exists('../data') else 'data'
PITCH_LENGTH = 105.0
PITCH_WIDTH = 68.0

print("Initializing Pitch Control Engine...")

## 1. Extract Instantaneous Tactical Snapshot
We sample active positions for Attacking Team (Team A) and Defending Team (Team B).

In [ ]:
# Synthetic / sampled tactical snapshot for demonstration
att_players = np.array([
    [25.0, 34.0],  # GK / Deep pivot
    [45.0, 15.0],  # LB
    [48.0, 28.0],  # LCB
    [50.0, 40.0],  # RCB
    [47.0, 55.0],  # RB
    [65.0, 25.0],  # LCM
    [68.0, 42.0],  # RCM
    [78.0, 34.0],  # CAM
    [85.0, 12.0],  # LW
    [88.0, 34.0],  # CF
    [86.0, 56.0]   # RW
])

def_players = np.array([
    [10.0, 34.0],  # GK
    [28.0, 16.0],  # RB
    [26.0, 28.0],  # RCB
    [26.0, 40.0],  # LCB
    [27.0, 52.0],  # LB
    [42.0, 22.0],  # RM
    [40.0, 34.0],  # CM
    [41.0, 46.0],  # LM
    [58.0, 30.0],  # SS
    [62.0, 38.0]   # ST
])

ball_pos = np.array([72.0, 35.0])
print(f"Tactical Snapshot: {len(att_players)} Attacking players, {len(def_players)} Defending players.")

## 2. Compute Time-to-Intercept & Pitch Control Probability Surface
Discretize the pitch into a grid ($50 \times 35$) and solve arrival time $T(x, y)$.

In [ ]:
def compute_pitch_control(att_pos, def_pos, grid_nx=50, grid_ny=35, v_max=5.0, t_reaction=0.7, tau=0.45):
    x_coords = np.linspace(0, PITCH_LENGTH, grid_nx)
    y_coords = np.linspace(0, PITCH_WIDTH, grid_ny)
    X, Y = np.meshgrid(x_coords, y_coords)
    
    grid_points = np.stack([X.ravel(), Y.ravel()], axis=1)
    
    # Compute minimum arrival time for attacking team
    # Distance: (N_grid, N_att)
    att_dist = np.linalg.norm(grid_points[:, None, :] - att_pos[None, :, :], axis=2)
    att_times = t_reaction + att_dist / v_max
    min_att_time = att_times.min(axis=1)
    
    # Compute minimum arrival time for defending team
    def_dist = np.linalg.norm(grid_points[:, None, :] - def_pos[None, :, :], axis=2)
    def_times = t_reaction + def_dist / v_max
    min_def_time = def_times.min(axis=1)
    
    # Relative time advantage
    time_diff = (min_def_time - min_att_time) / tau
    prob_att = 1.0 / (1.0 + np.exp(-time_diff))
    
    prob_surface = prob_att.reshape((grid_ny, grid_nx))
    return X, Y, prob_surface

X, Y, pc_surface = compute_pitch_control(att_players, def_players)
print(f"Pitch control surface computed. Attacking control mean: {pc_surface.mean():.2%}")

## 3. Visualizing Spatial Pitch Control Surface Overlay
Red = Attacking Control, Blue = Defending Control, Neutral = Contested.

In [ ]:
plt.figure(figsize=(11, 7))
contour = plt.contourf(X, Y, pc_surface, levels=25, cmap='RdBu_r', alpha=0.85, vmin=0, vmax=1)
plt.colorbar(contour, label='Pitch Control Probability ($P_{\text{Att}}$)')

# Draw pitch markings
plt.plot([0, 105, 105, 0, 0], [0, 0, 68, 68, 0], color='black', linewidth=1.5)
plt.axvline(52.5, color='black', linestyle='--', alpha=0.6)

# Plot players
plt.scatter(att_players[:, 0], att_players[:, 1], c='#d62728', s=120, edgecolors='black', label='Attacking Team')
plt.scatter(def_players[:, 0], def_players[:, 1], c='#1f77b4', s=120, edgecolors='black', label='Defending Team')
plt.scatter(ball_pos[0], ball_pos[1], c='yellow', s=100, marker='*', edgecolors='black', label='Ball Position')

plt.title('Spatial Pitch Control & Territorial Dominance Model', fontsize=14)
plt.xlabel('X (m)')
plt.ylabel('Y (m)')
plt.xlim(-1, 106)
plt.ylim(-1, 69)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()